# Nigeria DHS WASH harmonisation
### A Colab walkthrough for 2013, 2018 and 2024

**What you will make**  
A compressed household CSV and a short quality-control CSV from the three audited Household Recode files and their three Geographic Data ZIPs.

1. Open [Google Colab](https://colab.research.google.com/), choose **File → Upload notebook**, and select this `.ipynb`.
2. Choose **Runtime → Run all**, or run the cells in order. The first code cell shows six upload prompts. At each prompt click **Choose File**, navigate to any folder on your computer, and select the requested file. Keep the GPS archives zipped.
3. After the sixth file is accepted, the remaining cells read, harmonise, link and check the data. If running one cell at a time, continue to Step 2 when the first cell finishes.
4. At the end, open the **Files** sidebar, refresh it if needed, and open `wash_outputs` under `/content`. Use the three-dot menu beside each CSV to download it to your computer. Download before closing or resetting the runtime.

Keep the inputs and exports private under your DHS data-use terms. The household export includes survey identifiers and displaced cluster coordinates.

The notebook uses pandas and NumPy and reads the GPS archives with the Python standard library. A regular CPU runtime is sufficient.



## 1. Pick the six files from your computer

Pick the 2013 household file and GPS ZIP, then the 2018 pair, then the 2024 pair. You can choose a different folder at each prompt. A renamed file is accepted if its contents pass the checks for that slot. A wrong file triggers another prompt for the same slot. If you cancel, rerun this cell. It reuses files already accepted in the current session. A new runtime starts with empty storage.

The checks target the specific releases audited for this study. Select the Stata household data files (`.DTA` or `.dta`) and the Geographic Data ZIPs.

In [ ]:
from pathlib import Path
import math
import struct
from zipfile import BadZipFile, ZipFile

import numpy as np
import pandas as pd

CONFIG = {
    2013: {
        "hr": "NGHR6AFL.DTA", "zip": "NG_2013_DHS_11072026_2323_246081.zip",
        "ge": "NGGE6AFL", "state": "shstate", "sachet": 72,
        "households": 38522, "clusters": 896, "usable_clusters": 889,
        "usable_households": 38215, "interview_year_counts": {2013: 38522},
    },
    2018: {
        "hr": "NGHR7BFL.DTA", "zip": "NG_2018_DHS_11072026_2322_246081.zip",
        "ge": "NGGE7BFL", "state": "shstate", "sachet": 92,
        "households": 40427, "clusters": 1389, "usable_clusters": 1382,
        "usable_households": 40220, "interview_year_counts": {2018: 40427},
    },
    2024: {
        "hr": "NGHR8BFL.dta", "zip": "NG_2024_DHS_11072026_2318_246081.zip",
        "ge": "NGGE8AFL", "state": "shstate1", "sachet": 72,
        "households": 40047, "clusters": 1380, "usable_clusters": 1380,
        "usable_households": 40047, "interview_year_counts": {2023: 4902, 2024: 35145},
    },
}

try:
    from google.colab import files as colab_files
except ModuleNotFoundError:
    colab_files = None

if colab_files is None:
    # Local mode supports running the same notebook outside Colab.
    INPUT_DIR = Path("upload")
    OUTPUT_DIR = Path("outputs/wash_colab_run")
else:
    INPUT_DIR = Path("/content/wash_inputs")
    OUTPUT_DIR = Path("/content/wash_outputs")
    INPUT_DIR.mkdir(parents=True, exist_ok=True)

def check_input(path, year, kind, spec):
    """Check that the selected file belongs in the requested slot."""
    if kind == "hr":
        year_counts = pd.read_stata(
            path, columns=["hv007"], convert_categoricals=False
        ).hv007.value_counts(dropna=False).to_dict()
        if year_counts != spec["interview_year_counts"]:
            raise ValueError(f"Expected the audited {year} Household Recode file.")
    else:
        with ZipFile(path) as archive:
            names = set(archive.namelist())
            for member in names:
                if Path(member).name == spec["ge"] + ".dbf":
                    stem = member[:-4]
                    if {stem + ext for ext in (".shp", ".shx", ".dbf", ".prj")} <= names:
                        return
        raise ValueError(f"Expected the {year} Geographic Data ZIP containing {spec['ge']}.")

def pick_input(year, kind, spec, number):
    destination = INPUT_DIR / spec[kind]
    if destination.is_file():
        check_input(destination, year, kind, spec)
        print(f"[{number}/6] Already accepted: {destination.name}", flush=True)
        return

    label = "Household Recode (.DTA)" if kind == "hr" else "Geographic Data (ZIP)"
    staged = destination.with_suffix(destination.suffix + ".part")
    while True:
        print(f"\n[{number}/6] Choose {year} {label}: {spec[kind]}", flush=True)
        try:
            colab_files.upload_file(str(staged))
        except ValueError as exc:
            staged.unlink(missing_ok=True)
            raise RuntimeError("Upload interrupted. Rerun this cell to resume at this file.") from exc

        try:
            check_input(staged, year, kind, spec)
        except (ValueError, OSError, EOFError, struct.error, BadZipFile) as exc:
            staged.unlink(missing_ok=True)
            print(f"File did not pass this check: {exc} Please try again.", flush=True)
        else:
            staged.replace(destination)
            print(f"Accepted {year} {label}.", flush=True)
            return

for index, (year, spec) in enumerate(CONFIG.items()):
    for offset, kind in enumerate(("hr", "zip"), start=1):
        if colab_files is None:
            check_input(INPUT_DIR / spec[kind], year, kind, spec)
        else:
            pick_input(year, kind, spec, 2 * index + offset)

print("\nAll six inputs are ready.")
print("pandas", pd.__version__, "| NumPy", np.__version__)

## 2. Read the household files and standardise survey fields

A **record** is one interviewed household. Read only the columns needed here so Colab uses less memory. The **key** `survey_round × hv001 × hv002` distinguishes households across rounds. PSU and stratum identifiers are also made unique across rounds. This keeps the design fields available for survey analysis. Decisions about pooled survey weights depend on the estimand and are made later.

DHS state and zone numbers can mean different things across releases. Decode them from each file's own value labels. Keep `survey_round` separate from `interview_month`, because the 2024 release includes December 2023 interviews. Keep the original DHS values alongside the study variables.

In [ ]:
DESIGN = "hv001 hv002 hv005 hv006 hv007 hv012 hv015 hv021 hv022 hv023 hv024 hv025".split()
WASH_RAW = "hv201 hv202 hv235 hv204 hv205 hv225 hv238 hv230a hv230b hv232 hv232b hv232y".split()
ASSETS = {
    "hv206": "asset_electricity", "hv207": "asset_radio", "hv208": "asset_tv",
    "hv209": "asset_fridge", "hv210": "asset_bicycle", "hv211": "asset_motorcycle",
    "hv212": "asset_car", "hv243a": "asset_mobile_phone",
}
KEY = ["survey_round", "hv001", "hv002"]

def label_map(labels, field):
    if field in labels:
        return labels[field]
    if field.upper() in labels:
        return labels[field.upper()]
    raise ValueError(f"No Stata value labels found for {field}")

def clean_labels(values, labels, field):
    names = values.map(label_map(labels, field)).astype("string").str.strip().str.title()
    return names.replace({"Fct-Abuja": "FCT", "Fct Abuja": "FCT"})

def read_households(round_year, spec):
    source = INPUT_DIR / spec["hr"]
    with pd.read_stata(source, iterator=True) as reader:
        labels = reader.value_labels()
    state_field = spec["state"]
    columns = DESIGN + WASH_RAW + list(ASSETS) + ["hv219", "hv220", state_field]
    if round_year == 2024:
        columns += ["shstate", "hv201b"]
    df = pd.read_stata(source, columns=columns, convert_categoricals=False)
    df = df.loc[df.hv015.eq(1)].copy()
    df.insert(0, "survey_round", round_year)
    assert len(df) == spec["households"], f"{round_year}: unexpected completed household count"
    assert df.hv007.value_counts().to_dict() == spec["interview_year_counts"]
    assert df[KEY].notna().all().all() and not df.duplicated(KEY).any()
    assert df.hv001.nunique() == spec["clusters"]
    assert df.hv005.gt(0).all() and df.hv012.ge(0).all()
    assert df.hv006.between(1, 12).all()
    assert df[["hv021", "hv022", "hv023"]].notna().all().all()
    df["interview_year"] = df.hv007.astype(int)
    df["interview_month_number"] = df.hv006.astype(int)
    df["interview_month"] = pd.to_datetime({
        "year": df.interview_year, "month": df.interview_month_number, "day": 1,
    })
    df["state"] = clean_labels(df[state_field], labels, state_field)
    df["zone"] = clean_labels(df.hv024, labels, "hv024")
    df["residence"] = df.hv025.map({1: "Urban", 2: "Rural"}).astype("string")
    assert df.state.nunique() == 37 and df.zone.nunique() == 6
    assert df[["state", "zone", "residence"]].notna().all().all()
    df["hh_weight"] = df.hv005 / 1_000_000
    df["de_jure_weight"] = df.hh_weight * df.hv012
    df["cluster_key"] = str(round_year) + ":" + df.hv001.astype(str)
    df["psu_key"] = str(round_year) + ":" + df.hv021.astype(str)
    df["stratum_hv022_key"] = str(round_year) + ":" + df.hv022.astype(str)
    df["stratum_hv023_key"] = str(round_year) + ":" + df.hv023.astype(str)
    df["usual_household_size"] = df.hv012.astype("Int64")
    df["head_sex"] = df.hv219.map({1: "Male", 2: "Female"}).astype("string")
    df["head_age_years_capped"] = df.hv220.where(df.hv220.between(0, 97)).astype("Int64")
    top_code = 95 if round_year == 2013 else 97
    df["head_age_topcoded"] = df.head_age_years_capped.eq(top_code).astype("Int8")
    for raw, name in ASSETS.items():
        df[name] = df[raw].where(df[raw].isin([0, 1])).astype("Int8")
    asset_values = df[list(ASSETS.values())]
    df["nonwash_assets_observed"] = asset_values.notna().sum(axis=1).astype("Int8")
    df["nonwash_asset_count"] = asset_values.sum(axis=1, min_count=len(ASSETS)).astype("Int8")
    return df

households = {year: read_households(year, spec) for year, spec in CONFIG.items()}
print({year: len(df) for year, df in households.items()})

The asset count is a simple 0–8 summary, with missing values when any asset response is unavailable. The eight individual indicators are retained for later modelling. Household-head education would require the member recode files. The age field retains the released top code, with a flag identifying top-coded observations.

## 3. Apply a common WASH definition

Each service flag has three possible values. **1** means basic service, **0** means below basic, and a blank means the available answers cannot establish service status. Leave refusals and other non-observation unresolved. The earlier round has different hygiene codes and sachet water uses different codes across releases.

For drinking water, the default uses the released collection time and the improved-source definition consistently across rounds. In 2018 and 2024, bottled and sachet water households were routed through the question about water for other uses. The notebook therefore also gives you a sensitivity flag that leaves basic-water status unresolved for these routed cases. Treat unspecified source/facility code 96 as unresolved for the study indicators. Published benchmark rules are applied separately in Step 5.

The primary three-domain flag uses 2018 and 2024 and requires all three services to be classifiable. The all-round version is available for sensitivity analyses. A value of **1** indicates deprivation in at least two domains. The lower and upper bounds show what happens if unknown domains are assigned the best or worst status.

In [ ]:
IMPROVED_WATER_BASE = {11, 12, 13, 14, 21, 31, 41, 51, 61, 62, 71}
IMPROVED_SAN = {11, 12, 13, 15, 21, 22, 41}

def harmonise_wash(df, year, spec):
    improved = df.hv201.isin(IMPROVED_WATER_BASE | {spec["sachet"]})
    below_source = df.hv201.isin([32, 42, 43])
    time_basic = df.hv204.eq(996) | df.hv204.between(0, 30)
    time_long = df.hv204.between(31, 995)
    packaged = df.hv201.isin([71, spec["sachet"]])
    df["water_improved_source"] = pd.Series(pd.NA, index=df.index, dtype="Int8")
    df.loc[improved, "water_improved_source"] = 1
    df.loc[below_source, "water_improved_source"] = 0
    df["water_basic"] = pd.Series(pd.NA, index=df.index, dtype="Int8")
    df.loc[improved & time_basic, "water_basic"] = 1
    df.loc[below_source | (improved & time_long), "water_basic"] = 0
    df["water_time_other_use_routing"] = (packaged & (year in [2018, 2024])).astype("Int8")
    df["water_basic_strict_time"] = df.water_basic.mask(df.water_time_other_use_routing.eq(1))

    san_improved = df.hv205.isin(IMPROVED_SAN)
    df["sanitation_basic"] = pd.Series(pd.NA, index=df.index, dtype="Int8")
    df.loc[san_improved & df.hv225.eq(0), "sanitation_basic"] = 1
    df.loc[df.hv205.isin([14, 23, 31, 42, 43]) | (san_improved & df.hv225.eq(1)), "sanitation_basic"] = 0

    if year == 2013:
        status_map = {1: "Observed", 2: "Outside premises", 3: "Permission refused", 4: "Other non-observation"}
        observed = df.hv230a.eq(1)
        outside = df.hv230a.eq(2)
    else:
        status_map = {1: "Observed fixed", 2: "Observed mobile", 3: "Outside premises", 4: "Permission refused", 5: "Other non-observation"}
        observed = df.hv230a.isin([1, 2])
        outside = df.hv230a.eq(3)
    df["hygiene_observation_status"] = df.hv230a.map(status_map).fillna("Unlabelled or missing")
    df["hygiene_observation_unresolved"] = (~(observed | outside)).astype("Int8")
    df["hygiene_basic"] = pd.Series(pd.NA, index=df.index, dtype="Int8")
    df.loc[observed & df.hv230b.eq(1) & df.hv232.eq(1), "hygiene_basic"] = 1
    df.loc[outside | (observed & (df.hv230b.eq(0) | df.hv232.eq(0))), "hygiene_basic"] = 0

    for domain in ["water", "sanitation", "hygiene"]:
        df[f"{domain}_dep"] = (1 - df[f"{domain}_basic"]).astype("Int8")
    deps = df[["water_dep", "sanitation_dep", "hygiene_dep"]]
    df["wash_domains_known"] = deps.notna().sum(axis=1).astype("Int8")
    df["wash_dep_count"] = deps.sum(axis=1, min_count=3).astype("Int8")
    df["wash_dep_2plus_all_rounds"] = df.wash_dep_count.ge(2).astype("Int8")
    df["wash_dep_2plus"] = df.wash_dep_2plus_all_rounds.copy()
    if year == 2013:
        df["wash_dep_2plus"] = pd.Series(pd.NA, index=df.index, dtype="Int8")
    df["wash_dep_2plus_lower"] = deps.fillna(0).sum(axis=1).ge(2).astype("Int8")
    df["wash_dep_2plus_upper"] = deps.fillna(1).sum(axis=1).ge(2).astype("Int8")
    assert (df.wash_dep_2plus_lower <= df.wash_dep_2plus_upper).all()
    assert df.wash_dep_2plus_all_rounds.notna().eq(df.wash_domains_known.eq(3)).all()
    return df

for year, df in households.items():
    households[year] = harmonise_wash(df, year, CONFIG[year])
print("WASH indicators constructed for all three rounds.")

## 4. Attach GPS without losing households

A **left join** keeps every household even if its coordinates are unusable. Many households belong to one cluster. The merge checks that each household can match at most one GPS row in its survey round.

The helper reads the coordinate columns in the shapefile's DBF attribute table and checks them against its point geometry. It checks the ZIP and coordinate reference system and marks the `(0, 0)` placeholders as missing. Coordinates are displaced cluster locations. Use `gps_usable` to select households for spatial linkage and keep the full household file for survey-weighted description.

In [ ]:
def read_dhs_gps(year, spec):
    with ZipFile(INPUT_DIR / spec["zip"]) as archive:
        if archive.testzip() is not None:
            raise ValueError(f"{year}: corrupted GPS archive")
        dbf_name, = [n for n in archive.namelist() if Path(n).name.lower() == (spec["ge"] + ".dbf").lower()]
        stem = dbf_name[:-4]
        dbf = archive.read(dbf_name)
        shp = archive.read(stem + ".shp")
        projection = archive.read(stem + ".prj").decode("ascii")
    assert "WGS_1984" in projection, f"{year}: review coordinate reference system"

    n, header_size, row_size = struct.unpack_from("<IHH", dbf, 4)
    layout = {}
    start = 1
    for pos in range(32, header_size - 1, 32):
        name = dbf[pos:pos + 11].split(b"\0")[0].decode("ascii")
        width = dbf[pos + 16]
        layout[name] = slice(start, start + width)
        start += width
    fields = ["DHSCLUST", "LATNUM", "LONGNUM", "DHSYEAR", "DHSCC"]
    records = []
    for i in range(n):
        start = header_size + i * row_size
        row = dbf[start:start + row_size]
        assert len(row) == row_size and row[:1] == b" "
        records.append([row[layout[field]].strip().decode("ascii") for field in fields])
    gps = pd.DataFrame(records, columns=fields)
    for col in fields[:-1]:
        gps[col] = pd.to_numeric(gps[col], errors="raise")

    points = []
    pos = 100
    while pos < len(shp):
        _, words = struct.unpack_from(">ii", shp, pos)
        shape_type = struct.unpack_from("<i", shp, pos + 8)[0]
        assert shape_type == 1 and words == 10, f"{year}: expected point geometry"
        points.append(struct.unpack_from("<dd", shp, pos + 12))
        pos += 8 + 2 * words
    assert pos == len(shp) and len(points) == len(gps)
    assert all(
        math.isclose(x, lon, abs_tol=1e-6) and math.isclose(y, lat, abs_tol=1e-6)
        for (x, y), lon, lat in zip(points, gps.LONGNUM, gps.LATNUM)
    ), f"{year}: attribute coordinates differ from geometry"

    assert gps.DHSYEAR.eq(year).all() and gps.DHSCC.eq("NG").all()
    assert gps.DHSCLUST.notna().all() and not gps.DHSCLUST.duplicated().any()
    assert len(gps) == spec["clusters"]
    placeholder = gps.LATNUM.eq(0) & gps.LONGNUM.eq(0)
    in_extent = gps.LATNUM.between(4, 14) & gps.LONGNUM.between(2, 15)
    assert (placeholder | in_extent).all(), f"{year}: review unexpected coordinates"
    gps["gps_usable"] = ~placeholder
    gps["gps_archive"] = spec["ge"]
    gps["survey_round"] = year
    gps.loc[placeholder, ["LATNUM", "LONGNUM"]] = np.nan
    return gps.drop(columns=["DHSYEAR", "DHSCC"]).rename(columns={
        "DHSCLUST": "hv001", "LATNUM": "cluster_latitude", "LONGNUM": "cluster_longitude",
    })

for year, spec in CONFIG.items():
    gps = read_dhs_gps(year, spec)
    df = households[year]
    assert set(df.hv001) == set(gps.hv001), f"{year}: household and GPS cluster sets differ"
    joined = df.merge(gps, on=["survey_round", "hv001"], how="left", validate="many_to_one")
    assert len(joined) == len(df) and not joined.duplicated(KEY).any()
    assert int(joined.gps_usable.sum()) == spec["usable_households"]
    assert joined.loc[joined.gps_usable, "hv001"].nunique() == spec["usable_clusters"]
    households[year] = joined

harmonised = pd.concat(households.values(), ignore_index=True)
assert len(harmonised) == 118_996 and not harmonised.duplicated(KEY).any()
assert harmonised.gps_usable.sum() == 118_482
print(f"{len(harmonised):,} households retained. {harmonised.gps_usable.sum():,} have usable GPS.")

## 5. Reproduce published benchmarks as a separate check

These checks use each report's own numerator and denominator. A **household-weighted** percentage gives each interviewed household its survey weight. A **usual-resident-weighted** percentage multiplies that weight by household size. The benchmark for 2018 basic sanitation reproduces when the denominator is restricted to toilet or latrine users. The study's population-wide estimate includes people in households practising open defecation.

The 13 checks below must match the published one-decimal percentages before files are written. They validate the source data and historical replication rules. The study outcomes use the common recodes from Step 3.

In [ ]:
def weighted_pct(mask, weights, eligible):
    assert not mask.isna().any() and not eligible.isna().any()
    denominator = weights[eligible].sum()
    if denominator <= 0:
        raise ValueError("The requested weighted denominator is empty.")
    return 100 * weights[eligible & mask].sum() / denominator

def replicate_report(df, year):
    all_rows = pd.Series(True, index=df.index)
    hh, pop = df.hh_weight, df.de_jure_weight
    observed = df.hv230a.eq(1) if year == 2013 else df.hv230a.isin([1, 2])
    soap_water = observed & df.hv230b.eq(1) & df.hv232.eq(1)
    water_time = df.hv204.eq(996) | df.hv204.between(0, 30)
    improved_san = df.hv205.isin(IMPROVED_SAN) & df.hv225.eq(0)
    if year == 2013:
        return [
            ("2.1", "Improved source", 60.6, weighted_pct(df.hv201.isin([11,12,13,21,31,41,51,71]), hh, all_rows)),
            ("2.1", "Sachet water", 5.8, weighted_pct(df.hv201.eq(72), hh, all_rows)),
            ("2.2", "Improved sanitation, unshared", 30.1, weighted_pct(df.hv205.isin(IMPROVED_SAN - {15}) & df.hv225.eq(0), hh, all_rows)),
            ("2.6", "Handwashing facility observed", 39.5, weighted_pct(observed, hh, all_rows)),
            ("2.6", "Soap and water among observed", 26.3, weighted_pct(soap_water, hh, observed)),
        ]
    water_codes = IMPROVED_WATER_BASE | ({72} if year == 2024 else set())
    water = df.hv201.isin(water_codes) & water_time
    hygiene_eligible = df.hv230a.isin([1, 2, 3])
    if year == 2018:
        return [
            ("2.1.1", "Basic drinking water", 61.9, weighted_pct(water, pop, all_rows)),
            ("2.3.1", "Basic sanitation among toilet/latrine users", 44.4, weighted_pct(improved_san, pop, df.hv205.ne(31))),
            ("2.7", "Handwashing facility observed", 81.1, weighted_pct(observed, pop, all_rows)),
            ("2.7", "Basic hygiene in eligible population", 31.4, weighted_pct(soap_water, pop, hygiene_eligible)),
        ]
    return [
        ("16.2", "At least basic drinking water", 70.5, weighted_pct(water, pop, all_rows)),
        ("16.7", "At least basic sanitation", 49.1, weighted_pct(improved_san, pop, all_rows)),
        ("16.11", "Handwashing facility observed", 74.5, weighted_pct(observed, pop, all_rows)),
        ("16.11", "Basic hygiene in eligible population", 36.6, weighted_pct(soap_water, pop, hygiene_eligible)),
    ]

benchmarks = pd.DataFrame(
    [(year, *row) for year, df in households.items() for row in replicate_report(df, year)],
    columns=["survey_round", "report_table", "indicator", "published_pct", "reproduced_pct"],
)
benchmarks["pass"] = benchmarks.reproduced_pct.round(1).eq(benchmarks.published_pct)
print(benchmarks.round(2).to_string(index=False))
assert len(benchmarks) == 13 and benchmarks["pass"].all(), "Review benchmark discrepancies."

## 6. Review quality control and save your outputs

Each row in the QC file describes one survey round. The missingness columns are percentages of weighted households. The all-round three-domain completeness measure requires classifiable water, sanitation and hygiene status even in 2013. The final column counts how many historical benchmarks passed.

The compressed household CSV can be read directly in pandas with `pd.read_csv(path, parse_dates=["interview_month"])`. The CSV uses blanks for missing values. Set binary fields to nullable integers when needed after import. The first day in `interview_month` represents the month for climate linkage. It is a constructed date and does not give the actual interview day.

In [ ]:
rows = []
for year, df in households.items():
    w = df.hh_weight
    all_rows = pd.Series(True, index=df.index)
    rows.append({
        "survey_round": year,
        "households": len(df),
        "duplicate_household_keys": int(df.duplicated(KEY).sum()),
        "clusters": df.hv001.nunique(),
        "usable_gps_clusters": df.loc[df.gps_usable, "hv001"].nunique(),
        "usable_gps_households": int(df.gps_usable.sum()),
        "usable_gps_weighted_pct": weighted_pct(df.gps_usable, w, all_rows),
        "water_basic_missing_weighted_pct": weighted_pct(df.water_basic.isna(), w, all_rows),
        "sanitation_basic_missing_weighted_pct": weighted_pct(df.sanitation_basic.isna(), w, all_rows),
        "hygiene_basic_missing_weighted_pct": weighted_pct(df.hygiene_basic.isna(), w, all_rows),
        "hygiene_observation_unresolved_weighted_pct": weighted_pct(df.hygiene_observation_unresolved.eq(1), w, all_rows),
        "three_domains_complete_weighted_pct": weighted_pct(df.wash_domains_known.eq(3), w, all_rows),
        "report_benchmarks_passed": int(benchmarks.loc[benchmarks.survey_round.eq(year), "pass"].sum()),
    })

qc = pd.DataFrame(rows)
expected_gps_pct = [99.15, 99.46, 100.00]
assert qc.usable_gps_weighted_pct.round(2).tolist() == expected_gps_pct
assert qc.hygiene_observation_unresolved_weighted_pct.round(2).tolist() == [26.35, 7.86, 8.30]
assert qc.report_benchmarks_passed.sum() == 13
qc = qc.round(2)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
data_path = OUTPUT_DIR / "Nigeria_DHS_WASH_Households.csv.gz"
qc_path = OUTPUT_DIR / "Nigeria_DHS_WASH_Quality_Control.csv"
harmonised.to_csv(data_path, index=False, compression="gzip", date_format="%Y-%m-%d")
qc.to_csv(qc_path, index=False)

# Reopen the actual files and verify the exported keys and record counts.
reopened = pd.read_csv(data_path, usecols=KEY + ["gps_usable", "interview_month"], parse_dates=["interview_month"])
assert len(reopened) == 118_996 and not reopened.duplicated(KEY).any()
assert reopened.gps_usable.sum() == 118_482 and reopened.interview_month.notna().all()
pd.testing.assert_frame_equal(pd.read_csv(qc_path), qc)

print(qc.to_string(index=False))
print(f"\nHousehold data saved to {data_path}")
print(f"Quality control saved to {qc_path}")

### What to do with the outputs

Use all eligible household rows for weighted WASH summaries. Use `gps_usable == True` when creating a cluster-month climate exposure table. Join that later table by `survey_round`, `hv001`, and `interview_month`. The default study water indicator should be compared with `water_basic_strict_time` when you assess sensitivity to other-use routing.

**References**

- [Nigeria DHS WASH Harmonisation Matrix](https://chatgpt.com/library?file_id=libfile_9463c622e9b081919f03d92185aefb4e)
- [2013 NDHS final report](https://dhsprogram.com/pubs/pdf/FR293/FR293.pdf)
- [2018 NDHS final report](https://dhsprogram.com/pubs/pdf/FR359/FR359.pdf)
- [2024 NDHS final report](https://www.health.gov.ng/wp-content/uploads/2025/10/NDHS-2024-Final-Report_Disseminated17102025.pdf)
- [WHO/UNICEF JMP estimation methods](https://washdata.org/topics/methods/estimation-methods)